# Step 1: Environment setup and dependency installation

In [ ]:
!pip install torch torchvision transformers "datasets<4.0.0" pillow numpy tqdm matplotlib
!pip install mamba-ssm causal-conv1d>=1.2.0
!pip install "numpy<2.0.0" ipywidgets
!pip uninstall -y numpy
!pip install numpy==1.26.4
!pip install accelerate

In [ ]:
import torch

is_available = torch.cuda.is_available()
print("Is GPU available:", is_available)

if is_available:
    print("GPU Model:", torch.cuda.get_device_name(0))

# Step 2: Global Parameter Configuration and Data Processing

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datasets import load_dataset
from PIL import Image
from transformers import AutoTokenizer, ViTImageProcessor, ViTForImageClassification
from torchvision import models, transforms

# ===================== 1. Global Configuration =====================
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"  
else:
    DEVICE = "cpu"

print(f"Current compute device: {DEVICE}")
CONTEXT_LENGTHS = [1024, 4096, 8192, 16384]
NEEDLE_KEY = "Experiment Key: PG19_MEMORY_TEST_2026"
PG19_SPLIT = "test"
PLOT_SAVE_PATH = "memory_accuracy_curve.png"
plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] 
plt.rcParams['axes.unicode_minus'] = False

# ===================== 2. Load PG19 Long Text Dataset =====================
def load_pg19_long_text():
    print("Streaming PG19 dataset...")
    dataset = load_dataset("pg19", split=PG19_SPLIT, streaming=True, trust_remote_code=True)
    for sample in dataset:
        if len(sample["text"].split()) > 50000:
            long_text = sample["text"]
            break
    print(f"Original PG19 text length: {len(long_text.split()):,} words")
    return long_text

# ===================== 3. Construct Long Text Memory Test Data =====================
def build_memory_test_data(full_text, context_len, needle=NEEDLE_KEY):
    text_trunc = full_text[:context_len * 4]
    insert_pos = np.random.randint(100, len(text_trunc) // 2)
    test_text = text_trunc[:insert_pos] + needle + text_trunc[insert_pos:]
    return test_text, needle

# ===================== 4. Text-to-Image Tool (Adapts to Vision Models/MLP) =====================
class Text2Image:
    def __init__(self, img_size=(224, 224)):
        self.img_size = img_size
        self.transform = transforms.Compose([
            transforms.Resize(img_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        ])

    def convert(self, text):
        char_codes = [ord(c) % 256 for c in text]
        pixel_num = self.img_size[0] * self.img_size[1]
        if len(char_codes) < pixel_num:
            char_codes += [0] * (pixel_num - len(char_codes))
        else:
            char_codes = char_codes[:pixel_num]
        img_array = np.array(char_codes, dtype=np.uint8).reshape(self.img_size)
        img = Image.fromarray(img_array).convert("RGB")
        img_tensor = self.transform(img).unsqueeze(0).to(DEVICE)
        return img_tensor

# Step 3: Model Define

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# ===================== 5. Implementation of Memory Testers for Each Model =====================
import torch.nn.functional as F

EMBED_DIM = 768        
HIDDEN_DIM = 1024     
DROPOUT = 0.2
SEQ_MAX = 2048        

class SimpleMambaBlock(nn.Module):
    def __init__(self, d_model=EMBED_DIM, d_state=16):
        super().__init__()
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=4, padding=3, groups=d_model)
        self.x_proj = nn.Linear(d_model, d_state * 2 + d_model)
        self.dt_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, L, D = x.shape
        xz = self.in_proj(x)
        x_in, z = xz.chunk(2, dim=-1)
        x_conv = x_in.transpose(1, 2)
        x_conv = self.conv1d(x_conv)[:, :, :L].transpose(1, 2)
        x_conv = F.silu(x_conv)
        x_proj = self.x_proj(x_conv)
        delta, A, B_val = torch.split(x_proj, [D, 16, 16], dim=-1)
        delta = F.softplus(self.dt_proj(delta))
        y = x_conv * delta
        y = y * F.silu(z)
        return self.out_proj(y)

class PureMambaMemoryTester:
    def __init__(self, max_length=SEQ_MAX):
        print("Initializing Mamba model and tokenizer...")
        
        self.tokenizer = AutoTokenizer.from_pretrained("state-spaces/mamba-130m-hf")
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.model = AutoModel.from_pretrained(
            "state-spaces/mamba-130m-hf",
            torch_dtype=torch.bfloat16
        ).to(self.device).eval()

        self.max_length = max_length
        print(f"Initialization complete! Current device: {self.device}, Max truncation length: {self.max_length}")

    def test_memory(self, test_text, answer):
        with torch.no_grad():
            inputs = self.tokenizer(
                test_text,
                return_tensors="pt",
                truncation=True,
                max_length=self.max_length
            ).to(self.device)
        
            outputs = self.model(
                inputs.input_ids,
                output_hidden_states=True,
                return_dict=True
            )
            last_hidden = outputs.hidden_states[-1] 
    
            answer_inputs = self.tokenizer(
                answer,
                return_tensors="pt"
            ).to(self.device)
            answer_outputs = self.model(
                answer_inputs.input_ids,
                output_hidden_states=True,
                return_dict=True
            )
            answer_hidden = answer_outputs.hidden_states[-1].mean(dim=1)
        
            text_hidden = last_hidden.mean(dim=1)  
            
            similarity = torch.cosine_similarity(text_hidden, answer_hidden, dim=1).item()

            hit = similarity > 0.85
            
        return hit, similarity

In [ ]:
class ViTMemoryTester:
    def __init__(self):
        self.text2img = Text2Image()
        self.processor = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224")
        self.model = ViTForImageClassification.from_pretrained("google/vit-base-patch16-224").to(DEVICE).eval()

    def test_memory(self, test_text, answer):
        with torch.no_grad():
            img_text = self.text2img.convert(test_text)
            img_answer = self.text2img.convert(answer)
            
            feat_text = self.model.vit(img_text).last_hidden_state[:, 0, :].squeeze()
            feat_answer = self.model.vit(img_answer).last_hidden_state[:, 0, :].squeeze()

            similarity = torch.cosine_similarity(feat_text, feat_answer, dim=0).item()
            return similarity
        return hit

In [ ]:
class ResNetMemoryTester:
    def __init__(self):
        self.text2img = Text2Image()
        self.model = models.resnet50(pretrained=True).to(DEVICE).eval()
        self.feat_extractor = nn.Sequential(*list(self.model.children())[:-1]).eval()

    def test_memory(self, test_text, answer):
        with torch.no_grad():
            img_text = self.text2img.convert(test_text)
            img_answer = self.text2img.convert(answer)
            feat_text = self.feat_extractor(img_text).flatten()
            feat_answer = self.feat_extractor(img_answer).flatten()
            similarity = torch.cosine_similarity(feat_text, feat_answer, dim=0).item()
            return similarity
        return hit

In [ ]:
class MLPMemoryTester:
    def __init__(self):
        self.text2img = Text2Image()
        self.img_size = (224, 224)
        self.input_dim = 3 * self.img_size[0] * self.img_size[1]
        self.mlp = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.input_dim, HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_DIM, HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(HIDDEN_DIM, EMBED_DIM)
        ).to(DEVICE).eval()

    def test_memory(self, test_text, answer):
        with torch.no_grad():
            img_text = self.text2img.convert(test_text)
            img_answer = self.text2img.convert(answer)
            feat_text = self.mlp(img_text).squeeze()
            feat_answer = self.mlp(img_answer).squeeze()
            similarity = torch.cosine_similarity(feat_text, feat_answer, dim=0).item()
            return similarity
        return hit

# Step 4: Main Experimental Workflow

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# ===================== 6. Main Experiment Process + Result Visualization =====================
def run_experiment():
    pg19_text = load_pg19_long_text()
    
    print("\n========== Initialize All Models ==========")
    model_testers = {
        "PyTorch_Mamba": PureMambaMemoryTester(), 
        "ViT": ViTMemoryTester(),
        "ResNet50": ResNetMemoryTester(),
        "MLP": MLPMemoryTester()
    }
    
    accuracy_results = {name: [] for name in model_testers.keys()}
    
    num_trials = 20 
    print(f"\n========== Start Long Text Memory Test ({num_trials} independent trials per length) ==========")
    
    for length in tqdm(CONTEXT_LENGTHS, desc="Testing Context Length"):
        length_hits = {name: [] for name in model_testers.keys()}
        
        for i in range(num_trials):
            test_text, answer = build_memory_test_data(pg19_text, length)
            
            for model_name, tester in model_testers.items():
                hit = tester.test_memory(test_text, answer)
                length_hits[model_name].append(hit)
                
        for model_name in model_testers.keys():
            avg_hit = np.mean(length_hits[model_name])
            accuracy_results[model_name].append(avg_hit)

    plot_accuracy_curve(accuracy_results, CONTEXT_LENGTHS)
    
    print_report(accuracy_results, CONTEXT_LENGTHS)

# ===================== 7. Experiment Report Output =====================
def print_report(accuracy, context_lengths):
    print("\n\n========== PG19 Long Text Memory Test Experiment Report ==========")
    header = f"{'Model':<12}" + "".join([f"{l:<10}" for l in context_lengths]) + f"{'Avg Acc':<15}"
    print(header)
    print("-" * len(header))
    for model_name, acc_list in accuracy.items():
        avg_acc = np.mean(acc_list) * 100
        acc_str = "".join([f"{acc*100:<10.1f}%" for acc in acc_list])
        print(f"{model_name:<12}{acc_str}{avg_acc:<15.1f}%")
    print("-" * len(header))
    print(f"Test Task: Needle in a Haystack (Retrieval of a unique key randomly inserted in long text)")
    print(f"Device: {DEVICE} | Visualization results saved to: {PLOT_SAVE_PATH}")

# ===================== 8. Plot Accuracy Curve =====================
def plot_accuracy_curve(accuracy, context_lengths):
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
    markers = ["o", "s", "^", "x"]
    model_names = list(accuracy.keys())
    for i, name in enumerate(model_names):
        ax.plot(
            context_lengths,
            accuracy[name],
            label=name,
            color=colors[i],
            marker=markers[i],
            linewidth=2,
            markersize=8
        )
    ax.set_xlabel("Context Length (tokens)", fontsize=12, fontweight="bold")
    ax.set_ylabel("Cosine Similarity (Feature Match)", fontsize=12, fontweight="bold")
    ax.set_title("PG19 Long Text: Feature Similarity vs Context Length", fontsize=14, fontweight="bold", pad=20)
    ax.set_ylim(-0.2, 1.1)  
    ax.grid(True, alpha=0.3, linestyle="-")
    ax.legend(loc="upper right", fontsize=10)
    plt.tight_layout()
    plt.savefig(PLOT_SAVE_PATH, dpi=300, bbox_inches="tight")
    plt.close()

# ===================== Main Function Execution =====================
if __name__ == "__main__":
    torch.no_grad()  
    run_experiment()